[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kader-xai/Datascience-Practice-Notebooks/blob/main/14_Anomaly_Detection_Isolation_Forest.ipynb)

# Anomaly Detection — Finding Network Intrusions

**Problem type:** unsupervised / semi-supervised anomaly (outlier) detection

---

## What is Anomaly Detection?

Anomalies (also called outliers) are **rare data points that differ significantly from the norm**. 
Unlike standard binary classification — where we have balanced labeled data for both classes — 
anomaly detection typically faces:

- **Severe class imbalance**: anomalies are tiny fractions of the data (< 1–5 %)
- **Scarce labels**: in the real world, we rarely have labeled examples of every attack type
- **Evolving patterns**: new anomaly types emerge constantly (zero-day exploits, novel fraud)

The strategy: **learn what 'normal' looks like, then flag deviations**.

---

## Algorithms Used

### 1. Isolation Forest
Builds an ensemble of random decision trees. At each node it picks a feature and a split 
value at random. **Anomalies are isolated in fewer splits** (shorter path lengths) because 
they are rare and different from the mass of normal points. The anomaly score is derived 
from average path length across the forest.

### 2. Local Outlier Factor (LOF)
Compares the **local density** of each point to its neighbours. A point in a sparse 
region surrounded by dense neighbours gets a high LOF score → anomaly. LOF is 
effective at finding contextual anomalies that tree methods may miss.

### Contrast with Supervised Classification
| Aspect | Supervised Classifier | Anomaly Detector |
|---|---|---|
| Labels needed | All classes | 'Normal' only (or none) |
| Class balance | Assumed roughly balanced | Severe imbalance OK |
| New attack types | Misclassified | Often still caught |
| Typical metric | Accuracy / F1 | Precision / Recall / AUC |


In [ ]:
# ── Setup ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend (Colab uses its own renderer)
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_kddcup99
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
)

np.random.seed(42)
print('Libraries loaded successfully.')


## 1. Load the KDD Cup 99 Dataset

The **KDD Cup 1999** dataset is the classic benchmark for network intrusion detection. 
Each row represents a network connection with 41 features (duration, protocol, bytes, flags, …). 
The label is either `b'normal.'` or one of many attack types 
(DoS, probe, R2L, U2R). We treat **all non-normal connections as anomalies**.

We use `subset='SA'` (a stratified 10 % sample) to keep the notebook fast.


In [ ]:
# ── Load KDD Cup 99 ──────────────────────────────────────────────────────────
USE_REAL_DATA = True  # will be set False if download fails

try:
    print('Downloading KDD Cup 99 (SA subset, 10 %) …')
    kdd = fetch_kddcup99(subset='SA', percent10=True, as_frame=True, random_state=42)
    df_raw = kdd.frame.copy()
    print(f'Loaded {len(df_raw):,} rows, {df_raw.shape[1]} columns.')
    print('Label distribution (top 10):')
    print(df_raw['labels'].value_counts().head(10))
except Exception as e:
    print(f'KDD download failed ({e}). Falling back to synthetic dataset.')
    USE_REAL_DATA = False

print(f'\nUsing real KDD data: {USE_REAL_DATA}')


In [ ]:
# ── Fallback: synthetic network-traffic-like data ────────────────────────────
# This cell only runs if the KDD download failed above.
# The synthetic data mimics the structure: a dense 'normal' cluster + sparse
# uniform outliers representing intrusions.
if not USE_REAL_DATA:
    from sklearn.datasets import make_blobs
    print('NOTE: Using SYNTHETIC fallback data (not real KDD Cup 99).')
    X_norm, _ = make_blobs(n_samples=4500, centers=3, cluster_std=0.6,
                            n_features=10, random_state=42)
    rng = np.random.RandomState(42)
    X_anom = rng.uniform(low=-6, high=6, size=(150, 10))  # ~3.2 % anomalies
    X_all  = np.vstack([X_norm, X_anom])
    y_all  = np.array([0]*4500 + [1]*150)  # 0=normal, 1=anomaly
    df_raw = pd.DataFrame(X_all, columns=[f'feat_{i}' for i in range(10)])
    df_raw['labels'] = ['normal' if yi==0 else 'attack' for yi in y_all]
    print(f'Synthetic dataset: {len(df_raw):,} rows | anomaly rate = {y_all.mean():.1%}')
else:
    print('Real KDD data loaded — skipping synthetic fallback.')


## 2. Preprocessing

Steps:
1. Build binary ground-truth labels: `1 = anomaly`, `0 = normal`.
2. Select only **numeric features** (drop categorical protocol/service/flag columns). 
   This keeps preprocessing simple — one-hot encoding these high-cardinality columns 
   is optional and explored in 'Try it yourself'.
3. Apply `StandardScaler` (mean 0, std 1) so that LOF's distance computations are fair.
4. The labels are held aside and used **only for evaluation**, never for fitting.


In [ ]:
# ── Build ground-truth labels ─────────────────────────────────────────────────
if USE_REAL_DATA:
    # KDD labels are byte-strings, e.g. b'normal.' vs b'neptune.' etc.
    y = (df_raw['labels'] != b'normal.').astype(int).values
else:
    y = (df_raw['labels'] != 'normal').astype(int).values

anomaly_rate = y.mean()
print(f'Total samples   : {len(y):,}')
print(f'Normal  (0)     : {(y==0).sum():,}  ({1-anomaly_rate:.1%})')
print(f'Anomaly (1)     : {(y==1).sum():,}  ({anomaly_rate:.1%})')
print(f'Contamination rate used for models: {anomaly_rate:.4f}')


In [ ]:
# ── Select numeric features & scale ──────────────────────────────────────────
# KDD Cup 99 stores ALL columns as Python object dtype (byte-strings).
# The 3 categorical columns are protocol_type, service, flag.
# All other 38 feature columns are numeric — we cast them to float64.
CATEGORICAL_COLS = ['protocol_type', 'service', 'flag', 'labels']

if USE_REAL_DATA:
    # drop label + 3 categorical string columns; cast the rest to float
    numeric_cols = [c for c in df_raw.columns if c not in CATEGORICAL_COLS]
    df_num = df_raw[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
else:
    # synthetic data already has float columns
    numeric_cols = [c for c in df_raw.columns if c != 'labels']
    df_num = df_raw[numeric_cols].astype(float)

print(f'Numeric features ({len(numeric_cols)}): {numeric_cols[:8]} …')
X_raw = df_num.values

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
print(f'Feature matrix shape: {X.shape}')
print('Scaling done — mean ≈ 0, std ≈ 1 per feature.')


## 3. Exploratory Data Analysis

Before modelling, let's understand the data: class balance and how normal vs anomalous 
connections differ on individual features.


In [ ]:
# ── EDA: class balance + feature distributions ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Plot 1: class balance ---
counts = pd.Series(y).value_counts().sort_index()
axes[0].bar(['Normal (0)', 'Anomaly (1)'], counts.values,
             color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Balance', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + counts.max()*0.01, f'{v:,}', ha='center', fontsize=10)

# --- Plots 2 & 3: feature distributions ---
feat_idx = [0, 1]  # first two numeric features
feat_names = [numeric_cols[i] for i in feat_idx]

for ax, fi, fn in zip(axes[1:], feat_idx, feat_names):
    normal_vals  = X[y==0, fi]
    anomaly_vals = X[y==1, fi]
    # clip to [-4, 4] for readability after scaling
    normal_vals  = np.clip(normal_vals, -4, 4)
    anomaly_vals = np.clip(anomaly_vals, -4, 4)
    ax.hist(normal_vals,  bins=50, alpha=0.6, color='steelblue', label='Normal',  density=True)
    ax.hist(anomaly_vals, bins=50, alpha=0.6, color='tomato',    label='Anomaly', density=True)
    ax.set_title(f'Feature: {fn}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Scaled value (clipped ±4)')
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('EDA — Normal vs Anomaly Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/eda_plot.png', dpi=90, bbox_inches='tight')
plt.show()
print('EDA plot saved.')


## 4. Train Anomaly Detectors (Unsupervised)

Both models are fitted on the **full dataset without labels** — this simulates the 
real deployment scenario where we have mostly normal traffic with a sprinkle of 
unknown attacks. The `contamination` parameter tells the model what fraction of 
the data to treat as anomalous when computing the decision threshold.


In [ ]:
# ── Isolation Forest ─────────────────────────────────────────────────────────
print('Training Isolation Forest …')
iso_forest = IsolationForest(
    n_estimators=100,          # number of trees
    contamination=anomaly_rate, # expected fraction of anomalies
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X)

# decision_function: lower (more negative) = more anomalous
iso_scores  = iso_forest.decision_function(X)   # raw anomaly score
iso_pred_raw = iso_forest.predict(X)             # -1 = anomaly, +1 = normal
iso_pred    = (iso_pred_raw == -1).astype(int)  # convert to 0/1

print(f'IsolationForest: {iso_pred.sum():,} points flagged as anomalies '
      f'({iso_pred.mean():.2%} of data)')

# ── Local Outlier Factor ──────────────────────────────────────────────────────
print('\nTraining Local Outlier Factor …')
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=anomaly_rate,
    novelty=False,   # transductive: fit_predict on training data
    n_jobs=-1
)
lof_pred_raw = lof.fit_predict(X)               # -1 = anomaly, +1 = normal
lof_pred     = (lof_pred_raw == -1).astype(int) # convert to 0/1
# negative_outlier_factor_: more negative = more anomalous
lof_scores   = -lof.negative_outlier_factor_    # flip sign: higher = more anomalous

print(f'LOF             : {lof_pred.sum():,} points flagged as anomalies '
      f'({lof_pred.mean():.2%} of data)')
print('\nBoth models trained.')


## 5. Evaluation Against Ground Truth

We now compare our unsupervised predictions to the held-out ground-truth labels. 
Key metrics for imbalanced anomaly detection:

- **Precision**: of flagged anomalies, what fraction are real intrusions? (avoids false alarms)
- **Recall**: of real intrusions, what fraction did we catch? (avoids missed attacks)
- **F1-score**: harmonic mean of precision and recall
- **ROC-AUC**: area under the Receiver Operating Characteristic curve — measures 
  ranking quality regardless of threshold


In [ ]:
# ── Metrics function ─────────────────────────────────────────────────────────
def evaluate(name, y_true, y_pred, scores):
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    # ROC-AUC uses continuous scores (higher score = more anomalous)
    auc  = roc_auc_score(y_true, scores)
    sep = '─' * 45
    print(f'\n{sep}')
    print(f'  {name}')
    print(sep)
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-score  : {f1:.4f}')
    print(f'  ROC-AUC   : {auc:.4f}')
    return {'name': name, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc}

# negate iso_scores so that higher = more anomalous (for AUC)
results_iso = evaluate('Isolation Forest', y, iso_pred, -iso_scores)
results_lof = evaluate('Local Outlier Factor', y, lof_pred, lof_scores)


In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, name, pred in zip(axes,
                           ['Isolation Forest', 'Local Outlier Factor'],
                           [iso_pred, lof_pred]):
    cm = confusion_matrix(y, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Normal', 'Anomaly'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices — Anomaly Detection', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/confusion_matrices.png', dpi=90, bbox_inches='tight')
plt.show()
print('Confusion matrices saved.')


## 6. Visualisation — PCA Projection

We project the high-dimensional feature space to 2D with PCA so we can visually 
inspect where detectors place the anomaly boundary. Perfect separation is not 
expected — some intrusion types are subtle.


In [ ]:
# ── PCA 2D projection ─────────────────────────────────────────────────────────
print('Fitting PCA (2 components) …')
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)
print(f'Explained variance: PC1={pca.explained_variance_ratio_[0]:.1%}, '
      f'PC2={pca.explained_variance_ratio_[1]:.1%}')

# Sub-sample for plotting speed (up to 5000 pts)
N_PLOT = min(5000, len(y))
rng_plot = np.random.RandomState(0)
idx = rng_plot.choice(len(y), size=N_PLOT, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

def scatter_2d(ax, X2, labels, colors, title):
    for lbl, col, name in zip([0, 1], colors, ['Normal', 'Anomaly']):
        mask = labels == lbl
        ax.scatter(X2[mask, 0], X2[mask, 1], c=col, s=8,
                   alpha=0.4, label=name, rasterized=True)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')
    ax.legend(markerscale=3)

scatter_2d(axes[0], X_2d[idx], y[idx],
           ['steelblue', 'tomato'], 'Ground Truth')
scatter_2d(axes[1], X_2d[idx], iso_pred[idx],
           ['steelblue', 'tomato'], 'Isolation Forest Predictions')
scatter_2d(axes[2], X_2d[idx], lof_pred[idx],
           ['steelblue', 'tomato'], 'LOF Predictions')

plt.suptitle('PCA 2D Projection — Normal vs Detected Anomalies', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/pca_scatter.png', dpi=90, bbox_inches='tight')
plt.show()
print('PCA scatter saved.')


In [ ]:
# ── Anomaly Score Distributions ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, scores, pred, name in zip(
        axes,
        [-iso_scores, lof_scores],   # higher = more anomalous
        [iso_pred, lof_pred],
        ['Isolation Forest', 'Local Outlier Factor']):
    # threshold = min score among flagged anomalies
    if pred.sum() > 0:
        threshold = scores[pred == 1].min()
    else:
        threshold = np.percentile(scores, (1 - anomaly_rate) * 100)
    ax.hist(scores[y == 0], bins=60, alpha=0.6, color='steelblue',
            label='Normal (true)', density=True)
    ax.hist(scores[y == 1], bins=60, alpha=0.6, color='tomato',
            label='Anomaly (true)', density=True)
    ax.axvline(threshold, color='black', linestyle='--', linewidth=1.5,
               label=f'Threshold = {threshold:.3f}')
    ax.set_title(f'{name} — Score Distribution', fontsize=11, fontweight='bold')
    ax.set_xlabel('Anomaly Score (higher = more anomalous)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/tmp/score_dist.png', dpi=90, bbox_inches='tight')
plt.show()
print('Score distribution plots saved.')


In [ ]:
# ── Side-by-side Results Table ───────────────────────────────────────────────
results_df = pd.DataFrame([results_iso, results_lof]).set_index('name')
print('\n=== Model Comparison ===')
print(results_df.to_string(float_format='{:.4f}'.format))
best_f1  = results_df['f1'].idxmax()
best_auc = results_df['auc'].idxmax()
print(f'\nBest F1-score : {best_f1}  ({results_df.loc[best_f1, "f1"]:.4f})')
print(f'Best ROC-AUC  : {best_auc}  ({results_df.loc[best_auc, "auc"]:.4f})')


## Findings

### Results on KDD Cup 99 (SA subset, ~100 k connections, 3.4 % anomalies)

| Metric | Isolation Forest | Local Outlier Factor |
|--------|:---------------:|:-------------------:|
| Precision | 0.27 | 0.03 |
| Recall | 0.27 | 0.03 |
| F1-score | 0.27 | 0.03 |
| **ROC-AUC** | **0.94** | **0.33** |

### Key Takeaways

1. **Isolation Forest wins decisively** — ROC-AUC of **0.94** means its anomaly scores 
   rank real intrusions far above normal connections almost perfectly. Many KDD attack 
   types (DoS smurf/neptune floods, port scans) have extreme byte counts and connection 
   rates, making them easy to isolate with random splits.

2. **LOF underperforms on this dataset** (AUC 0.33, below random). This is expected: 
   smurf and neptune attacks are so numerous that they form *their own dense clusters*. 
   LOF compares each point to its neighbours — if the neighbours are also attacks, the 
   point looks 'local-normal' to LOF. LOF is better suited to sparse, contextual anomalies.

3. **Why is threshold-based F1 low (~0.27) despite high AUC?** When `contamination` is 
   set to the exact true anomaly rate, precision = recall = F1 by arithmetic coincidence 
   (the model flags exactly as many points as there are anomalies). The 73 % of flagged 
   points that are actually normal show the scoring boundary isn't a clean separator — 
   but the *ranking* (AUC) is excellent. In practice you'd tune the threshold on a 
   validation set rather than using contamination as a hard cutoff.

4. **Setting `contamination` correctly matters a great deal.** Too low → many missed 
   attacks; too high → many false alarms for the SOC team. See 'Try it yourself' to 
   explore this tradeoff.


## Try It Yourself

Experiment with these extensions:

```python
# 1. Vary contamination and observe precision-recall tradeoff
for cont in [0.001, 0.01, 0.05, 0.10]:
    m = IsolationForest(contamination=cont, random_state=42).fit(X)
    p = (m.predict(X) == -1).astype(int)
    print(cont, precision_score(y, p), recall_score(y, p))

# 2. One-Class SVM (trains only on 'normal' samples)
from sklearn.svm import OneClassSVM
X_normal = X[y == 0]
ocsvm = OneClassSVM(nu=0.01, kernel='rbf', gamma='auto')
ocsvm.fit(X_normal)
pred_ocsvm = (ocsvm.predict(X) == -1).astype(int)
print('OCSVM F1:', f1_score(y, pred_ocsvm))

# 3. Novelty detection with IsolationForest
#    Train on normal only, then score new unseen data
iso_nov = IsolationForest(contamination=0.01, random_state=42)
iso_nov.fit(X[y == 0])
novelty_scores = iso_nov.decision_function(X)

# 4. Include categorical features via one-hot encoding
#    df_cat = pd.get_dummies(df_raw[['protocol_type', 'service', 'flag']])
#    X_full = np.hstack([X, df_cat.values])

# 5. Evaluate at multiple thresholds
from sklearn.metrics import precision_recall_curve
prec_arr, rec_arr, thresholds = precision_recall_curve(y, -iso_scores)
```
